## Configuración: SIN trim en ambos años (decisión del 2026-09-17)

Para que la comparación sea válida, los dos años corren bajo el **mismo tratamiento**: sin
recorte iterativo y con `ing_cor` como denominador (la única variable de ingreso con
definición idéntica en 2014 y 2022).

Las cifras de bienestar de aquí **no coinciden** con las de `aradillas_2014.ipynb`, que es la
réplica de fidelidad al Gauss (con trim) y sirve de benchmark contra el paper. Ver el §5 del
reporte de hallazgos.

# Comparación 2014 ↔ 2022

Corre los dos años bajo **tratamiento idéntico** para que las diferencias observadas sean
atribuibles al cambio en la economía y no a la configuración del estimador.


> **Actualizado (5 ago 2026):** incorpora las cuatro correcciones de fidelidad al Gauss en
> la ruta de bienestar — V1 umbral de significancia t ≥ 2.326 (l.6376), V2 markup de Lerner
> −1/ε para el contrafactual (l.6345, 6380), V3 denominador `ing_cor`, V4 medianas en vez de
> medias (l.6520). V2 y V4 **interactúan**: evaluados por separado dan una lectura invertida.

---

## Por qué existe este notebook

`aradillas_2014.ipynb` y `aradillas_2022.ipynb` corren cada año en su **mejor**
configuración, que no es la misma:

* **2014 con trim** — el recorte del 1 % por cola mejora el ajuste (MAE 0.217 contra 0.286
  sin trim) porque con 8,940 hogares recorta colas que efectivamente son ruido.
* **2022 sin trim** — con 57,552 hogares ese mismo recorte colapsa la varianza del regresor
  de utilidad y deja al 76 % de los hogares sin efectos ingreso identificados.

Esa asimetría es correcta para reportar **niveles** de cada año, pero inutiliza cualquier
afirmación del tipo *"el impuesto subió de X a Y"*: la diferencia confundiría el cambio
económico con el cambio de tratamiento.

**Aquí se corren ambos años sin trim y con el mismo denominador de ingreso.**

### Las dos decisiones de comparabilidad

**Trim: desactivado en ambos.** Es el único ajuste que permite tratar los dos años igual.
Tiene un costo, y hay que declararlo: la réplica 2014 sin trim es peor (MAE 0.217 → 0.286).
Se acepta porque el objetivo aquí es comparar, no replicar.

**Ingreso: `ing_cor` en ambos.** Es la única variable de ingreso con definición idéntica en
los dos concentrados. `ing_mon` no sirve para comparar: la ENIGH 2022 "Nueva serie" dejó de
publicarlo y hubo que reconstruirlo, con una razón `ing_mon/ing_cor` de 0.794 en 2014 contra
0.877 en 2022 — imposible saber si esa brecha es cambio real o diferencia de concepto.

**Lo que este notebook NO reemplaza:** el Gini de 2014 sobre `ing_mon` (0.481, idéntico al
publicado) sigue siendo el resultado de la réplica, y vive en `aradillas_2014.ipynb`.

---

### Advertencia sobre magnitudes en pesos

La VE de 2022 equivale al 77 % del gasto en categorías contra ~53 % en 2014, sin validar.
**Los porcentajes, las elasticidades y los β_η son sólidos; las cifras en pesos no.**

In [1]:
import sys
sys.path.insert(0, "Replica_COFECE/Codigo")
import numpy as np

import datos_2014, datos_2022
from aradillas_core import (estimar_easi, reconstruir_matrices, ModeloEASI,
                            demandas_marshallianas, elasticidades,
                            estimar_markups, variacion_equivalente,
                            cuadro_10, gini, sectores_significativos)


def correr(mod, ruta, etiqueta):
    """Pipeline completo bajo tratamiento comparable: sin trim, ingreso corriente."""
    d = mod.cargar(ruta, verbose=False)
    r = estimar_easi(d.precios_ln, d.w, d.gasto_total, d.Z, n_cat=d.n_cat,
                     aplicar_trim=False, verbose=False)
    d = d.submuestra(r["mask"])
    modelo = ModeloEASI(**reconstruir_matrices(r["beta"], n_cat=d.n_cat))
    util = modelo.utilidad_indirecta(d.precios_ln, d.Z, r["epsilon"], d.w,
                                     d.gasto_total, verbose=False)
    dem, _ = demandas_marshallianas(modelo, d.precios_ln, d.Z, util, r["epsilon"],
                                    d.gasto_total, d.factor_expansion, d.n_cat)
    e_nac, e_cd = elasticidades(modelo, d.precios_ln, d.Z, r["epsilon"], d.w,
                                d.gasto_total, d.factor_expansion, dem, d.ciudad,
                                d.n_ciudades, d.n_cat, util,
                                nombres=d.nombres_cat, verbose=False)
    mk = estimar_markups(d.precios_por_ciudad(), e_cd, d.vars_costos)
    sig = sectores_significativos(mk["t_eta"], mk["beta_eta"])   # V1: t >= 2.326
    VE = variacion_equivalente(modelo, d.precios_ln, d.Z, r["epsilon"], d.w,
                               d.gasto_total, mk["markup_lerner"][d.ciudad], sig,
                               verbose=False)                      # V2: Lerner
    c10 = cuadro_10(VE, d.ingreso_cor)          # V3 ing_cor + V4 mediana
    g = gini(d.ingreso_cor_completo, c10["tasas"])
    print(f"{etiqueta}: {d.n_hogares} hogares, {int(sig.sum())}/{d.n_cat} sectores signif.")
    return dict(d=d, e=np.abs(e_nac), mk=mk, sig=sig, c10=c10, g=g)


r14 = correr(datos_2014, "Replica_COFECE/Data_2014", "2014")
r22 = correr(datos_2022, "Replica_COFECE/Data_2022/", "2022")

2014: 12372 hogares, 9/12 sectores signif.


2022: 57552 hogares, 9/13 sectores signif.


## Elasticidades

In [2]:
# Se empareja por NOMBRE: 2022 tiene categorías que 2014 no (Pan de caja, P1),
# así que la posición j ya no identifica la misma categoría en ambos años.
cats = r14["d"].nombres_cat
j22 = {n: k for k, n in enumerate(r22["d"].nombres_cat)}
solo22 = [n for n in r22["d"].nombres_cat if n not in cats]
print(f"{'Categoría':<22} {'2014':>8} {'2022':>8} {'cambio':>9}")
print("-"*50)
for j, n in enumerate(cats):
    a, b = r14["e"][j], r22["e"][j22[n]]
    print(f"  {n:<20} {a:>8.3f} {b:>8.3f} {b-a:>+9.3f}")
e22_comunes = np.array([r22["e"][j22[n]] for n in cats])
print(f"\n{'promedio (comunes)':<22} {r14['e'].mean():>8.3f} {e22_comunes.mean():>8.3f}"
      f" {e22_comunes.mean()-r14['e'].mean():>+9.3f}")
for n in solo22:
    print(f"  {n:<20} {'—':>8} {r22['e'][j22[n]]:>8.3f}   (solo 2022)")

Categoría                  2014     2022    cambio
--------------------------------------------------
  Tortillas               0.809    0.780    -0.029
  Pan                     1.460    1.423    -0.038
  Pollo+Huevo             1.181    1.333    +0.153
  Carne res               1.623    0.653    -0.970
  Carnes proc.            1.453    0.997    -0.455
  Lácteos                 1.301    1.115    -0.185
  Frutas                  1.376    1.378    +0.003
  Verduras                1.106    1.133    +0.027
  Bebidas                 1.442    1.679    +0.237
  Medicamentos            1.008    0.772    -0.236
  Transporte foráneo      0.169    0.489    +0.320
  Materiales              0.609    0.878    +0.269

promedio (comunes)        1.128    1.053    -0.075
  Pan de caja                 —    1.237   (solo 2022)


## Poder de mercado (β_η)

Un β_η mayor indica mayor capacidad de fijar precio por encima del costo marginal.

In [3]:
print(f"{'Categoría':<22} {'β 2014':>8} {'t':>7} {'β 2022':>8} {'t':>7} {'cambio':>9}")
print("-"*66)
for j, n in enumerate(cats):
    k = j22[n]
    b14, t14 = r14["mk"]["beta_eta"][j], r14["mk"]["t_eta"][j]
    b22, t22 = r22["mk"]["beta_eta"][k], r22["mk"]["t_eta"][k]
    marca = ""
    if r22["sig"][k] and not r14["sig"][j]:
        marca = "  ← gana significancia"
    elif r14["sig"][j] and not r22["sig"][k]:
        marca = "  ← pierde significancia"
    print(f"  {n:<20} {b14:>8.3f} {t14:>7.2f} {b22:>8.3f} {t22:>7.2f} {b22-b14:>+9.3f}{marca}")
for n in solo22:
    k = j22[n]
    print(f"  {n:<20} {'—':>8} {'—':>7} {r22['mk']['beta_eta'][k]:>8.3f}"
          f" {r22['mk']['t_eta'][k]:>7.2f}   (solo 2022)")
print(f"\nsectores significativos: {int(r14['sig'].sum())} en 2014, "
      f"{int(r22['sig'].sum())} en 2022")

Categoría                β 2014       t   β 2022       t    cambio
------------------------------------------------------------------
  Tortillas               0.119    2.15    0.326    3.73    +0.207  ← gana significancia
  Pan                     1.011    9.20    1.373    9.31    +0.361
  Pollo+Huevo             0.268    2.71    0.115    2.17    -0.153  ← pierde significancia
  Carne res               0.403    4.16    0.158    5.74    -0.245
  Carnes proc.            0.197    1.52    0.207    5.59    +0.010  ← gana significancia
  Lácteos                 0.726    5.53    0.090    1.54    -0.636  ← pierde significancia
  Frutas                  0.966    7.86    0.752    7.30    -0.214
  Verduras                0.502    7.77   -0.006   -0.12    -0.508  ← pierde significancia
  Bebidas                 0.272    3.71    0.324    3.87    +0.051
  Medicamentos            0.310    3.29    0.549    9.45    +0.239
  Transporte foráneo      0.032    2.42    0.023    0.42    -0.009  ← pierde sig

## Bienestar: incidencia por decil

Ambos años sobre `ing_cor`, así que las cifras son directamente comparables.

In [4]:
print(f"{'Decil':<7} {'2014 %':>9} {'2022 %':>9} {'cambio':>9}")
print("-"*38)
for f14, f22 in zip(r14["c10"]["deciles"], r22["c10"]["deciles"]):
    print(f"  {f14['decil']:<5} {f14['pct']:>9.1f} {f22['pct']:>9.1f}"
          f" {f22['pct']-f14['pct']:>+9.1f}")
t14, t22 = r14["c10"]["total"]["pct"], r22["c10"]["total"]["pct"]
print(f"  {'Tot':<5} {t14:>9.1f} {t22:>9.1f} {t22-t14:>+9.1f}")
print(f"\nRegresividad D1/D10:  2014 = {r14['c10']['regresividad']:.2f}"
      f"   2022 = {r22['c10']['regresividad']:.2f}")
print(f"\nGini observado:       2014 = {r14['g']['observado']:.3f}"
      f"   2022 = {r22['g']['observado']:.3f}")
print(f"Gini contrafactual:   2014 = {r14['g']['contrafactual']:.3f}"
      f"   2022 = {r22['g']['contrafactual']:.3f}")
print(f"Reducción del Gini:   2014 = {r14['g']['reduccion_pct']:.1f}%"
      f"   2022 = {r22['g']['reduccion_pct']:.1f}%")

Decil      2014 %    2022 %    cambio
--------------------------------------
  1           8.6      11.1      +2.5
  2           6.1       8.7      +2.6
  3           5.1       8.0      +2.9
  4           4.3       7.4      +3.1
  5           3.6       6.7      +3.1
  6           3.2       6.0      +2.8
  7           2.6       5.4      +2.8
  8           2.3       4.7      +2.4
  9           1.4       3.8      +2.4
  10          1.0       2.4      +1.4
  Tot         3.2       5.9      +2.7

Regresividad D1/D10:  2014 = 8.90   2022 = 4.61

Gini observado:       2014 = 0.433   2022 = 0.409
Gini contrafactual:   2014 = 0.425   2022 = 0.398
Reducción del Gini:   2014 = 2.0%   2022 = 2.9%


## Lectura

Las diferencias de esta tabla son atribuibles al cambio en la economía entre 2014 y 2022,
**no** a la configuración del estimador — que es idéntica en ambas corridas.

Al interpretar, tener presente:

* La ENIGH 2022 tiene 90,102 hogares contra 19,124 en 2014. Las tasas de retención tras los
  filtros son casi iguales (64.4 % y 65.8 %), así que la muestra es comparable en
  composición, pero los errores estándar de 2022 son mecánicamente menores.
* Sin trim, el 34 % de los hogares de 2022 conserva efectos ingreso débilmente identificados
  (contra 6.9 % en 2014 con trim). Las categorías con más desviación —Bebidas y Transporte
  foráneo— son candidatas a estar afectadas por eso y **no deberían reportarse como cambio
  económico sin verificación adicional**.
* Las magnitudes en pesos no están validadas; usar porcentajes.